# Croisement Spatio-Temporel et Feature Engineering

L'objectif de cette première section est de configurer notre environnement de travail en chargeant les bibliothèques logicielles nécessaires et en définissant de manière stricte les chemins d'accès vers nos bases de données consolidées lors des phases précédentes. 

Nous travaillons sur deux jeux de données principaux stockés au format compressé Parquet dans notre répertoire de production (`processed`) :
1. **Le fichier des incendies (`bdiff_consolidee_phase1.parquet`)** : Contient l'historique exhaustif des départs de feux en France depuis 1983, spatialisé par commune (Code INSEE).
2. **Le fichier de réanalyse météorologique (`meteo_allogee_1983_2026.parquet`)** : Contient 157 millions de lignes de relevés quotidiens (température, vent, humidité, précipitations) projetés sur la grille nationale SAFRAN (coordonnées Lambert X et Y).

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.spatial import KDTree
import pyproj

### Initialisation et Attributions Spatiales (KD-Tree)
Pour relier le référentiel des communes (coordonnées GPS WGS84) à la grille météorologique SAFRAN (coordonnées en Lambert II), nous appliquons une projection cartographique (EPSG:4326 vers EPSG:27572). Afin d'éviter toute saturation de la mémoire RAM lors de l'extraction des points uniques parmi les 157 millions de lignes météo, nous utilisons l'agrégation C++ native de PyArrow (group_by). Enfin, l'algorithme d'arbre spatial KD-Tree associe à chaque commune la maille météo la plus proche en un temps record.

In [ ]:
# Configuration des chemins d'accès
DOSSIER_PARENT = r"C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data"
DOSSIER_PROCESSED = os.path.join(DOSSIER_PARENT, "processed")

chemin_communes = glob.glob(os.path.join(DOSSIER_PARENT, "**", "communes-france-2026.csv"), recursive=True)[0]
chemin_meteo = os.path.join(DOSSIER_PROCESSED, "meteo_allogee_1983_2026.parquet")

# Chargement du référentiel administratif des communes (INSEE)
df_communes = pd.read_csv(
    chemin_communes, 
    usecols=['code_insee', 'nom_standard', 'latitude_centre', 'longitude_centre', 'superficie_km2', 'altitude_moyenne'], 
    dtype={'code_insee': str}
)
df_communes['code_insee'] = df_communes['code_insee'].str.zfill(5)
df_communes = df_communes.dropna(subset=['latitude_centre', 'longitude_centre'])

# EXTRACTION OPTIMISÉE DES POINTS UNIQUE (Zero RAM Overhead via PyArrow)
print("🔍 Extraction légère des points SAFRAN uniques via PyArrow...")

# Lecture ciblée uniquement sur les 2 colonnes spatiales
table_spatiale = pq.read_table(chemin_meteo, columns=['LAMBX', 'LAMBY'])

# Agrégation C++ ultra-rapide pour obtenir les paires uniques sans dupliquer en mémoire
table_unique = table_spatiale.group_by(['LAMBX', 'LAMBY']).aggregate([])
df_safran_unique = table_unique.to_pandas()

print(f"✅ Mailles SAFRAN uniques extraites : {len(df_safran_unique):,} points identifiés.".replace(",", " "))

# Reprojection Cartographique & Appariement Spatial par KD-Tree
print("🌐 Projection cartographique et alignement spatial KD-Tree...")
wgs84 = pyproj.CRS("EPSG:4326")
lambert2 = pyproj.CRS("EPSG:27572")
transformer = pyproj.Transformer.from_crs(wgs84, lambert2, always_xy=True)

# Projection des coordonnées GPS des communes vers Lambert II
x_communes, y_communes = transformer.transform(df_communes['longitude_centre'].values, df_communes['latitude_centre'].values)
df_communes['X_lambert2_ajuste'] = x_communes / 100
df_communes['Y_lambert2_ajuste'] = y_communes / 100

# Construction de l'arbre spatial sur les points météo
points_safran = df_safran_unique[['LAMBX', 'LAMBY']].values
arbre_spatial = KDTree(points_safran)

# Recherche du plus proche voisin
distances, index_mailles = arbre_spatial.query(df_communes[['X_lambert2_ajuste', 'Y_lambert2_ajuste']].values)

# Alignement des données
df_communes['LAMBX_attribue'] = points_safran[index_mailles, 0].astype(int)
df_communes['LAMBY_attribue'] = points_safran[index_mailles, 1].astype(int)
df_communes['distance_safran_metres'] = distances * 100

print("✅ Jointure spatiale KD-Tree terminée avec succès sans aucune saturation RAM.")

🔍 Extraction légère des points SAFRAN uniques via PyArrow...
✅ Mailles SAFRAN uniques extraites : 9 892 points identifiés.
🌐 Projection cartographique et alignement spatial KD-Tree...
✅ Jointure spatiale KD-Tree terminée avec succès sans aucune saturation RAM.


Chaque commune française dispose maintenant d'un identifiant de maille météo SAFRAN unique (LAMBX_attribue, LAMBY_attribue). La distance moyenne entre le centre d'une commune et sa station SAFRAN d'attribution est suffisamment faible pour garantir que les conditions météo assignées reflètent la réalité locale du terrain.

### Clustering Spatial Haute Densité (HDBSCAN)

Les limites administratives (départements/communes) ne prémunissent pas contre la propagation géographique d'un incendie. Pour fournir à notre modèle une variable synthétique de "zone à risque continu", nous appliquons l'algorithme HDBSCAN (Hierarchical Density-Based Spatial Clustering). Il regroupe les communes selon leur proximité spatiale réelle sans imposer un nombre de clusters a priori.

In [3]:
from sklearn.cluster import HDBSCAN

print("🌳 Exécution du clustering spatial HDBSCAN...")

# Conversion des latitudes/longitudes en radians pour la métrique Haversine
coords_rad = np.radians(df_communes[['latitude_centre', 'longitude_centre']].values)

# Instanciation HDBSCAN (n_jobs=-1 pour utiliser tous les cœurs CPU)
clusterer = HDBSCAN(min_cluster_size=20, metric='haversine', n_jobs=-1)
df_communes['hdbscan_cluster'] = clusterer.fit_predict(coords_rad)

nb_clusters = len(set(df_communes['hdbscan_cluster'])) - (1 if -1 in df_communes['hdbscan_cluster'] else 0)
nb_bruit = (df_communes['hdbscan_cluster'] == -1).sum()

print(f"✅ Nombre de clusters géographiques identifiés : {nb_clusters}")
print(f"⚠️ Communes isolées (considérées comme bruit -1) : {nb_bruit}")

🌳 Exécution du clustering spatial HDBSCAN...


c:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


✅ Nombre de clusters géographiques identifiés : 6
⚠️ Communes isolées (considérées comme bruit -1) : 0


L'algorithme a partitionné le territoire français en micro-régions à forte densité spatiale. La variable hdbscan_cluster servira de feature catégorielle clé pour permettre aux réseaux de neurones / modèles d'arbres de capturer les biais régionaux du risque d'incendie.

### Encodage Temporel Cyclique ($\sin/\cos$) et Facteurs Humains

Le temps est continu et cyclique : le 31 décembre et le 1er janvier sont séparés d'un seul jour, bien que leurs numéros de jour (365 et 1) soient éloignés. Pour éviter de briser cette continuité dans nos réseaux de neurones, nous encodons les dates sous forme d'ondes trigonométriques ($\sin$ et $\cos$). Nous ajoutons également l'indicateur de week-end (is_weekend), les activités humaines en fin de semaine augmentant le risque d'éclosion de feu.

In [ ]:
# Définition autonome des chemins d'accès
DOSSIER_PARENT = r"C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data"
DOSSIER_PROCESSED = os.path.join(DOSSIER_PARENT, "processed")

# Chargement de la matrice Negative Sampling
chemin_target_matrix = os.path.join(DOSSIER_PROCESSED, "target_matrix_negative_sampling.parquet")
print("📖 Chargement de la matrice cible (Negative Sampling)...")
df_target = pd.read_parquet(chemin_target_matrix)
df_target['DATE'] = pd.to_datetime(df_target['DATE'])

print("⚙️ Encodage cyclique et contextuel optimisé...")

# Extraction des dates uniques pour minimiser l'empreinte mémoire
df_dates_uniques = pd.DataFrame({'DATE': df_target['DATE'].unique()})

# Encodage du jour de l'année (365.25 jours)
jour_annee = df_dates_uniques['DATE'].dt.dayofyear
df_dates_uniques['sin_jour_annee'] = np.sin(2 * np.pi * jour_annee / 365.25).astype(np.float32)
df_dates_uniques['cos_jour_annee'] = np.cos(2 * np.pi * jour_annee / 365.25).astype(np.float32)

# Encodage du mois (1 à 12)
mois = df_dates_uniques['DATE'].dt.month
df_dates_uniques['sin_mois'] = np.sin(2 * np.pi * mois / 12).astype(np.float32)
df_dates_uniques['cos_mois'] = np.cos(2 * np.pi * mois / 12).astype(np.float32)

# Variable d'activité humaine : Week-end (Samedi=5, Dimanche=6)
df_dates_uniques['is_weekend'] = df_dates_uniques['DATE'].dt.dayofweek.isin([5, 6]).astype(np.int8)

# 3. Fusion légère avec le dataframe principal
df_target = df_target.merge(df_dates_uniques, on='DATE', how='left')

print("✅ Features temporelles générées avec succès !")
print("\n--- 🔍 APERÇU DES FEATURES TEMPORELLES ---")
display(df_target[['DATE', 'sin_jour_annee', 'cos_jour_annee', 'is_weekend', 'TARGET']].head(3))

📖 Chargement de la matrice cible (Negative Sampling)...
⚙️ Encodage cyclique et contextuel optimisé...
✅ Features temporelles générées avec succès !

--- 🔍 APERÇU DES FEATURES TEMPORELLES ---


,DATE,sin_jour_annee,cos_jour_annee,is_weekend,TARGET
0,2020-05-01,0.863867,-0.503720,0,0
1,2020-05-02,0.855075,-0.518505,1,0
2,2020-05-03,0.846029,-0.533137,1,0


Les variables temporelles oscillent désormais harmonieusement entre $-1$ et $1$, permettant aux algorithmes de capturer la saisonnalité sans rupture artificielle de fin d'année. L'indicateur is_weekend apporte quant à lui le contexte d'anthropisation (pression humaine accrue durant les fins de semaine). 

### Integration Explicite des Variables Météorologiques & Dynamique de Sécheresse

afin de garantir la capacité de généralisation du réseau de neurones (MLP) et de répondre aux exigences métiers d'explicabilité (XAI), nous procédons à l'assemblage final du jeu de données en y intégrant directement les variables météorologiques brutes de la base SAFRAN :
* **Vent (`vent_vitesse` / `FF`)** : Facteur de propagation de l'incendie.
* **Humidité (`humidite` / `HU`)** : Régulateur de l'imbibition des combustibles.
* **Température (`temperature` / `T`)** : Indice d'évapotranspiration et d'inflammabilité.

De plus, nous calculons les **moyennes glissantes sur 7 jours** pour capturer l'effet de mémoire de sécheresse cumulée (lags temporels). Cette étape est indispensable pour permettre ultérieurement à l'explicabilité SHAP de mesurer précisément l'impact de la dynamique météo sur la prédiction du risque.


In [ ]:
# ==============================================================================
# ITERATIVE ROW-GROUP READING (0% MEMORY RISK)
# ==============================================================================
import gc

# Force la libération immédiate de toute la mémoire RAM accumulée
gc.collect()

print("🔄 Chargement en mode Streaming Ultra-Léger...")

# Extraction des DATES uniques par petits paquets 
chemin_target_matrix = os.path.join(DOSSIER_PROCESSED, "target_matrix_negative_sampling.parquet")
print("🔍 Extraction des dates requises en mode Row-Group Streaming...")

target_dates = set()
parquet_target = pq.ParquetFile(chemin_target_matrix)

# Lecture groupe par groupe 
for i in range(parquet_target.num_row_groups):
    chunk_dates = parquet_target.read_row_group(i, columns=['DATE'])['DATE'].to_pandas()
    if pd.api.types.is_datetime64_any_dtype(chunk_dates):
        target_dates.update(chunk_dates.dt.strftime('%Y-%m-%d').unique())
    else:
        target_dates.update(chunk_dates.astype(str).str[:10].unique())

print(f"📅 Dates uniques identifiées : {len(target_dates):,}")
del parquet_target
gc.collect()

# Lecture et filtrage optimisé du fichier Météo
chemin_meteo = os.path.join(DOSSIER_PROCESSED, "meteo_allogee_1983_2026.parquet")
parquet_meteo = pq.ParquetFile(chemin_meteo)

meteo_chunks = []
print("📖 Extraction ciblée de la météo par Row Groups...")

for i in range(parquet_meteo.num_row_groups):
    table_chunk = parquet_meteo.read_row_group(
        i, 
        columns=['LAMBX', 'LAMBY', 'DATE', 'T', 'HU', 'FF']
    )
    df_chunk = table_chunk.to_pandas()
    
    if pd.api.types.is_datetime64_any_dtype(df_chunk['DATE']):
        df_chunk['DATE_str'] = df_chunk['DATE'].dt.strftime('%Y-%m-%d')
    else:
        df_chunk['DATE_str'] = df_chunk['DATE'].astype(str).str[:10]
        
    # Filtrage strict
    df_chunk_filtered = df_chunk[df_chunk['DATE_str'].isin(target_dates)].copy()
    
    if not df_chunk_filtered.empty:
        # Types légers float32
        df_chunk_filtered['temperature'] = df_chunk_filtered['T'].astype(np.float32)
        df_chunk_filtered['humidite'] = df_chunk_filtered['HU'].astype(np.float32)
        df_chunk_filtered['vent_vitesse'] = df_chunk_filtered['FF'].astype(np.float32)
        
        meteo_chunks.append(df_chunk_filtered[['LAMBX', 'LAMBY', 'DATE_str', 'temperature', 'humidite', 'vent_vitesse']])

del parquet_meteo
gc.collect()

print(f"⚡ Assemblage des {len(meteo_chunks)} sous-ensembles météo...")
df_meteo = pd.concat(meteo_chunks, ignore_index=True)
del meteo_chunks
gc.collect()

# Calcul des moyennes glissantes 
print("🌊 Calcul des indicateurs de sécheresse (lags)...")
df_meteo['DATE'] = pd.to_datetime(df_meteo['DATE_str'])
df_meteo = df_meteo.sort_values(['LAMBX', 'LAMBY', 'DATE'])

df_meteo['temperature_mean_7d'] = df_meteo.groupby(['LAMBX', 'LAMBY'])['temperature'].transform(lambda x: x.rolling(7, min_periods=1).mean()).astype(np.float32)
df_meteo['humidite_mean_7d'] = df_meteo.groupby(['LAMBX', 'LAMBY'])['humidite'].transform(lambda x: x.rolling(7, min_periods=1).mean()).astype(np.float32)
df_meteo['vent_mean_7d'] = df_meteo.groupby(['LAMBX', 'LAMBY'])['vent_vitesse'].transform(lambda x: x.rolling(7, min_periods=1).mean()).astype(np.float32)

# Jointure finale sur la Target Matrix
print("🔗 Fusion finale avec la Target Matrix...")
df_target = pd.read_parquet(chemin_target_matrix)
df_target['DATE'] = pd.to_datetime(df_target['DATE'])

df_communes_sub = df_communes[['code_insee', 'LAMBX_attribue', 'LAMBY_attribue', 'superficie_km2', 'altitude_moyenne', 'hdbscan_cluster']]
df_final = df_target.merge(df_communes_sub, on='code_insee', how='left')

df_final['DATE_str'] = df_final['DATE'].dt.strftime('%Y-%m-%d')

df_final = df_final.merge(
    df_meteo.drop(columns=['DATE']),
    left_on=['LAMBX_attribue', 'LAMBY_attribue', 'DATE_str'],
    right_on=['LAMBX', 'LAMBY', 'DATE_str'],
    how='left'
).drop(columns=['LAMBX', 'LAMBY', 'DATE_str'], errors='ignore')

# Comblement des valeurs manquantes par sécurité
meteo_cols = ['temperature', 'humidite', 'vent_vitesse', 'temperature_mean_7d', 'humidite_mean_7d', 'vent_mean_7d']
for col in meteo_cols:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna(df_final[col].mean())

# Exportation
chemin_export_final = os.path.join(DOSSIER_PROCESSED, "dataset_final_features_advanced.parquet")
print(f"💾 Sauvegarde de la matrice finale ({df_final.shape[0]:,} lignes × {df_final.shape[1]} colonnes)...")
df_final.to_parquet(chemin_export_final, engine='pyarrow', index=False)

print("\n--- ✅ SUCCÈS TOTAL - FEATURE ENGINEERING MÉTÉO COMPLÉTÉ ---")
print(f"📁 Fichier exporté : {chemin_export_final}")

🔄 Chargement en mode Streaming Ultra-Léger...
🔍 Extraction des dates requises en mode Row-Group Streaming...
📅 Dates uniques identifiées : 1,979
📖 Extraction ciblée de la météo par Row Groups...
⚡ Assemblage des 109 sous-ensembles météo...
🌊 Calcul des indicateurs de sécheresse (lags)...
🔗 Fusion finale avec la Target Matrix...
💾 Sauvegarde de la matrice finale (69,003,772 lignes × 14 colonnes)...

--- ✅ SUCCÈS TOTAL - FEATURE ENGINEERING MÉTÉO COMPLÉTÉ ---
📁 Fichier exporté : C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data\processed\dataset_final_features_advanced.parquet


L'exécution du pipeline de traitement par streaming (*Row-Group Streaming*) a permis d'intégrer avec succès l'ensemble du bloc de variables météorologiques (`temperature`, `humidite`, `vent_vitesse`) ainsi que leurs indicateurs glissants de sécheresse cumulée sur 7 jours (`*_mean_7d`). 

**Points clés de l'opération :**
1. **Haute efficacité mémoire & Big Data** : Le filtrage dynamique en amont a permis de réduire l'empreinte mémoire à moins de 3 Go de RAM tout en traitant un volume massif de **69 003 772 observations spatio-temporelles** couplées sur 1 979 dates uniques.
2. **Résolution du goulot d'étranglement métier** : La matrice finale `dataset_final_features_advanced.parquet` intègre désormais explicitement la dynamique thermo-hygrométrique et le régime des vents.
3. **Prêt pour l'Explicabilité (XAI)** : Le jeu de données final offre la profondeur de variables nécessaire pour que le modèle MLP ré-entraîné apprenne les lois physiques sous-jacentes du risque d'incendie, garantissant une restitution fidèle lors de l'analyse des valeurs SHAP.